### DATA INGESTION MODULE

### Workflows


1. **Config.yaml**: Este archivo contendrá toda la configuración relacionada con el proyecto. Se utilizará para almacenar todos los parámetros relacionados con el proyecto, así como todas las rutas del proyecto. También se usará para guardar todos los hiperparámetros.
2. **Params.yaml**: Este archivo contendrá todos los hiperparámetros relacionados con el proyecto. Se utilizará para almacenar los hiperparámetros del proyecto y, específicamente, los hiperparámetros del entrenamiento del modelo.
3. **Entidad de configuración (Config entity)**: Este componente definirá la estructura de los archivos de configuración y proporcionará una forma de acceder a los parámetros de configuración de manera estructurada.
4. **Administrador de configuración (Configuration Manager)**: Este componente gestionará la carga y actualización de los archivos de configuración, asegurando que se utilice la configuración correcta en todo el proyecto.
5. **Actualizar componentes**: Ingesta de datos, Transformación de datos, Entrenador del modelo.
6. **Crear pipelines**: Pipeline de entrenamiento, Pipeline de predicción.
7. **Frontend**: APIs, APIs de entrenamiento, APIs de predicción por lotes.

In [1]:
# Revisamos el directorio en el que nos enconrtamos.
import os
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer\\research'

In [2]:
# Vamos al directorio raíz del proyecto.
os.chdir("..")
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer'

In [3]:
# Creamos la entidad de configuración para la ingesta de datos.
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL_train: str
    source_URL_test: str
    source_URL_val: str

In [4]:
# Definimos en base a la información necesaria para pasarle la información a DataIngestionConfig en config.yaml.
'''
data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL_train: https://huggingface.co/datasets/knkarthick/samsum/resolve/main/train.csv
  source_URL_test: https://huggingface.co/datasets/knkarthick/samsum/resolve/main/test.csv
  source_URL_val: https://huggingface.co/datasets/knkarthick/samsum/resolve/main/validation.csv

'''


# Ahora importamos las constantes, que definimos en base a la información necesaria para pasarle la información a DataIngestionConfig.
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [5]:
# Creamos el configuration mangarer, que se encargará de cargar el archivo de configuración y devolver la configuración de cada componente.
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # Creamos el directorio de Artifacts, que es donde se guardarán todos los artefactos del proyecto.
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        # Creamos el directorio de ingesta de datos (A la función siempre le pasamos una lista)
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(root_dir=config.root_dir,
                                                    source_URL_train=config.source_URL_train,
                                                    source_URL_test=config.source_URL_test,
                                                    source_URL_val=config.source_URL_val)
        return data_ingestion_config

In [6]:
# Importamos las librerias para ingestar los datos y descargar los datos.
import os
import requests
from src.textSummarizer.logging import logger


# Componentes del módulo de ingesta de datos

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self, url: str, file_name: str) -> None:
        file_path = os.path.join(self.config.root_dir, file_name)
        logger.info(f"Descargando el archivo desde la URL: {url} a la ruta: {file_path}")
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa

        with open(file_path, "wb") as file:
            file.write(response.content)
        logger.info(f"Archivo descargado exitosamente en: {file_path}")

In [8]:
CONFIG_FILE_PATH

WindowsPath('C:/Users/mecenturion/Desktop/MARTIN/UDEMY/MLOPS/TextSummarizer/textsummarizer/config/config.yaml')

In [9]:
config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_file(url=data_ingestion_config.source_URL_train, file_name="train.csv")
data_ingestion.download_file(url=data_ingestion_config.source_URL_test, file_name="test.csv")
data_ingestion.download_file(url=data_ingestion_config.source_URL_val, file_name="val.csv")

[2026-04-14 10:44:09,812: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\config\config.yaml' read successfully.]
[2026-04-14 10:44:09,816: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\params.yaml' read successfully.]
[2026-04-14 10:44:09,818: INFO: common: 53: Directory 'artifacts' created successfully.]
[2026-04-14 10:44:09,823: INFO: common: 53: Directory 'artifacts/data_ingestion' created successfully.]
[2026-04-14 10:44:09,825: INFO: 356545463: 7: Descargando el archivo desde la URL: https://huggingface.co/datasets/knkarthick/samsum/resolve/main/train.csv a la ruta: artifacts/data_ingestion\train.csv]
[2026-04-14 10:44:10,995: INFO: 356545463: 13: Archivo descargado exitosamente en: artifacts/data_ingestion\train.csv]
[2026-04-14 10:44:11,000: INFO: 356545463: 7: Descargando el archivo desde la URL: https://huggingface.co/datasets/knkarthick/samsum/resolve/main/